# Choveu mais ou menos que o normal? — versão Databricks

Mesmo problema e mesmo dado da versão Colab, agora do jeito que se faz em produção:
arquivos no **Volume** do Unity Catalog, camadas **bronze → silver → gold** em **Delta**,
agregação com **PySpark** e pronto para virar **Job agendado**.

**Fonte do dado:** INMET, Dados Históricos (estações automáticas) — https://portal.inmet.gov.br/dadoshistoricos

Testado na Databricks Free Edition com compute serverless.

> Nota de projeto: o arquivo do INMET é "sujo" (8 linhas de cabeçalho, `;`, decimal vírgula,
> latin-1). Cada estação é pequena, então fazemos o *parse* inicial com pandas no driver e
> deixamos o Spark cuidar do armazenamento (Delta), da escala e do agendamento. É uma escolha
> comum quando a origem é um arquivo legado pequeno por chave.

## 0. Configuração
Os *widgets* aparecem no topo do notebook. Se o seu catálogo tiver outro nome, confira em
**Catalog** no menu lateral.

In [ ]:
dbutils.widgets.text("catalogo", "workspace")
dbutils.widgets.text("schema", "cases_agro")
dbutils.widgets.text("volume", "brutos")
dbutils.widgets.text("estacao", "A904")
dbutils.widgets.text("ano_inicio", "2021")
dbutils.widgets.text("ano_alvo", "2026")

CAT = dbutils.widgets.get("catalogo")
SCH = dbutils.widgets.get("schema")
VOL = dbutils.widgets.get("volume")
ESTACAO = dbutils.widgets.get("estacao")
ANO_INICIO = int(dbutils.widgets.get("ano_inicio"))
ANO_ALVO = int(dbutils.widgets.get("ano_alvo"))

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CAT}.{SCH}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CAT}.{SCH}.{VOL}")
CAMINHO_VOLUME = f"/Volumes/{CAT}/{SCH}/{VOL}"
print("Arquivos brutos em:", CAMINHO_VOLUME)

## 1. Ingestão — baixar os .zip anuais para o Volume
Um arquivo por ano, com todas as estações do Brasil.

**Se o download falhar** (alguns workspaces restringem a internet): baixe os `.zip` dos anos
pelo link do INMET e envie em **Catalog → cases_agro → brutos → Upload to this volume**.
Depois rode a próxima célula normalmente.

In [ ]:
import os
import requests

URL = "https://portal.inmet.gov.br/uploads/dadoshistoricos/{ano}.zip"
anos = list(range(ANO_INICIO, ANO_ALVO + 1))

for ano in anos:
    destino = f"{CAMINHO_VOLUME}/inmet_{ano}.zip"
    if os.path.exists(destino):
        continue
    try:
        r = requests.get(URL.format(ano=ano), timeout=600)
        r.raise_for_status()
        with open(destino, "wb") as f:
            f.write(r.content)
        print("Baixado:", destino)
    except Exception as e:
        print(f"Não consegui baixar {ano}:", e, "-> faça o upload manual para o Volume.")

## 2. Bronze — a chuva de hora em hora da estação, em Delta
Lemos só a estação escolhida de cada ano (parse com pandas), juntamos e gravamos em Delta.
Delta é como um caderno de campo com data em cada anotação: dá para voltar e ver versões.

In [ ]:
import io
import zipfile
import pandas as pd
from pyspark.sql import functions as F

# O INMET mudou o cabeçalho ao longo dos anos (a coluna de data é "DATA (YYYY-MM-DD)" nos
# arquivos antigos e "Data" nos novos). Por isso detectamos a coluna de data pela posição
# (1ª coluna) e a de precipitação pelo texto, em vez de fixar o nome.
def ler_estacao(ano):
    caminho = f"{CAMINHO_VOLUME}/inmet_{ano}.zip"
    with zipfile.ZipFile(caminho) as z:
        nomes = [n for n in z.namelist() if f"_{ESTACAO}_" in n]
        if not nomes:
            return None
        with z.open(sorted(nomes)[-1]) as f:
            bruto = io.BytesIO(f.read())
    pdf = pd.read_csv(bruto, sep=";", decimal=",", encoding="latin-1", skiprows=8)
    col_data = pdf.columns[0]
    col_prec = next(c for c in pdf.columns if "PRECIPITA" in c.upper())
    pdf = pdf[[col_data, col_prec]].rename(columns={col_data: "data_txt", col_prec: "precip_mm"})
    pdf["data_txt"] = pdf["data_txt"].astype(str)
    pdf["ano"] = ano
    return pdf[["ano", "data_txt", "precip_mm"]]

partes = [p for p in (ler_estacao(a) for a in anos) if p is not None]
pdf_tudo = pd.concat(partes, ignore_index=True)

bronze = spark.createDataFrame(pdf_tudo)
bronze.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CAT}.{SCH}.bronze_chuva")
display(spark.table(f"{CAT}.{SCH}.bronze_chuva").limit(10))

## 3. Silver — limpo e tipado
Mesmas regras da versão Colab: `precip_mm` vira número, `-9999` (sem medição) vira nulo e a
data vira data de verdade. Guardamos o mês para agrupar depois.

In [ ]:
silver = (spark.table(f"{CAT}.{SCH}.bronze_chuva")
          .withColumn("precip_mm", F.col("precip_mm").cast("double"))
          .withColumn("precip_mm", F.when(F.col("precip_mm") < 0, None).otherwise(F.col("precip_mm")))
          # entende os dois formatos de data do INMET: "2026/01/01" e "2016-01-01"
          .withColumn("data", F.coalesce(F.to_date("data_txt", "yyyy/MM/dd"),
                                         F.to_date("data_txt", "yyyy-MM-dd")))
          .withColumn("mes", F.month("data"))
          .select("ano", "data", "mes", "precip_mm"))
silver.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CAT}.{SCH}.silver_chuva")
display(spark.table(f"{CAT}.{SCH}.silver_chuva").limit(10))

## 4. Gold — a resposta: 2026 × média histórica, por mês
Somamos a chuva por ano e mês; a média histórica é a média de cada mês nos anos anteriores
ao alvo. Juntamos com o ano-alvo e calculamos a anomalia. Só ficam os meses já fechados em 2026.

In [ ]:
mensal = (spark.table(f"{CAT}.{SCH}.silver_chuva")
          .groupBy("ano", "mes").agg(F.sum("precip_mm").alias("chuva_mm")))

ref = (mensal.filter(F.col("ano") < ANO_ALVO)
       .groupBy("mes").agg(F.round(F.avg("chuva_mm"), 1).alias("media_historica_mm")))
alvo = (mensal.filter(F.col("ano") == ANO_ALVO)
        .select("mes", F.round("chuva_mm", 1).alias(f"ano_{ANO_ALVO}_mm")))

gold = (alvo.join(ref, "mes", "left")
        .withColumn("anomalia_mm", F.round(F.col(f"ano_{ANO_ALVO}_mm") - F.col("media_historica_mm"), 1))
        .orderBy("mes"))
gold.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CAT}.{SCH}.gold_anomalia_chuva")
display(spark.table(f"{CAT}.{SCH}.gold_anomalia_chuva"))

## 5. Visualizar
A tabela gold é pequena: convertemos para pandas só para desenhar o gráfico.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

MESES = ["Jan", "Fev", "Mar", "Abr", "Mai", "Jun", "Jul", "Ago", "Set", "Out", "Nov", "Dez"]
pdf = spark.table(f"{CAT}.{SCH}.gold_anomalia_chuva").toPandas()
rotulos = [MESES[int(m) - 1] for m in pdf["mes"]]

x = np.arange(len(pdf))
fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(x - 0.21, pdf["media_historica_mm"], width=0.42, label=f"Média {ANO_INICIO}–{ANO_ALVO-1}")
ax.bar(x + 0.21, pdf[f"ano_{ANO_ALVO}_mm"], width=0.42, label=str(ANO_ALVO))
ax.set_xticks(x); ax.set_xticklabels(rotulos)
ax.set_ylabel("chuva no mês (mm)"); ax.legend()
ax.set_title(f"Choveu mais ou menos que o normal? Estação {ESTACAO}")
plt.show()

## 6. (Este case não tem modelo)
Não há ML aqui — é uma comparação descritiva. Se um dia você quiser *prever* a chuva do mês,
aí entra o MLflow para registrar parâmetros, métricas e o modelo com versão no Unity Catalog.

## 7. Virar rotina (Job)
Para atualizar isto todo mês sem depender de lembrar:
1. Menu **Jobs & Pipelines → Create → Job**.
2. Tipo de tarefa: **Notebook**, escolha este notebook.
3. Em **Schedule**, defina mensal (ex.: todo dia 5).

**Diferença para o Colab:** aqui o dado fica governado no catálogo, versionado (Delta, dá para
auditar cada atualização) e a rotina roda sozinha. O mesmo resultado numérico da versão Colab.